# 19 · Product lookup and external ingredient metadata

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Product/ingredient information is not a chemical assay. Access and source licences require review.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Lookup one user-selected packaged product
No automatic product writes, image uploads, or bulk scraping.

In [ ]:
from oncoplate.external import lookup_product,nutrition5k_metadata,map_external_ingredients
BARCODE=''  # Enter an actual product barcode from your approved collection.
if BARCODE:
    snapshot=lookup_product(BARCODE,p['raw']/'open_food_facts','OncoPlateResearch/3.0 (anasstudyai@gmail.com)')
    print(snapshot)
else:
    print('No product requested. Supply a real barcode to run this optional lookup.')

## 2. Parse Nutrition5k dish metadata from an authorised local download
Use only overlapping ingredient tasks. Keep incremental scans of the same plate grouped.

In [ ]:
NUTRITION_ROOT=Path(cfg['root'])/'data/external/nutrition5k'
metadata_path=NUTRITION_ROOT/'metadata/dish_metadata_cafe1.csv'
assert metadata_path.exists(),'Place the real Nutrition5k metadata here; see docs/DATA_SOURCES.md. No records are fabricated.'
dishes,ingredients=nutrition5k_metadata(metadata_path)
write_table(p['reports']/'nutrition5k_dish_inventory.csv',dishes)
print('Dishes:',len(dishes),'Ingredient rows:',len(ingredients))

## 3. Apply only an independently reviewed taxonomy crosswalk

In [ ]:
crosswalk=read_table(NUTRITION_ROOT/'reviewed_ingredient_crosswalk.csv')
mapped=map_external_ingredients(ingredients,crosswalk)
write_table(p['private']/'nutrition5k_mapped_ingredients.csv',mapped)
print('Unmapped ingredient rows:',int((~mapped.mapped).sum()))
print('Unmapped ingredients stay unknown. No cooking style or cancer label is invented.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
